Menyalakan SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("Pertemuan7-StudiKasusTerintegrasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/11 10:03:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/11 10:03:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


In [ ]:
Menyiapkan Data Set

In [3]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


A. EXTRACT

In [5]:
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

print("Transaksi:", df_transaksi.count(), "|Cabang: ", df_cabang.count(), "|Menu: ", df_menu.count())

Transaksi: 4000 |Cabang:  3 |Menu:  15


B. TRANSFORM — Join & Kolom Turunan

In [6]:
df_gabungan = df_transaksi.join(df_cabang, on= "cabang_id", how= "inner").join(df_menu, on="menu_id", how= "inner")
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))
df_gabungan.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


C. TRANSFORM--WINDOW FUNCTION

In [7]:
ringkasan_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu").agg(spark_sum("total_penjualan").alias("total_penjualan"))
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())
top2_menu = ringkasan_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)).filter(col("peringkat")<=2)
top2_menu.orderBy("nama_cabang", "peringkat").show(truncate=False)

+-------------------------+---------+---------------+---------+
|nama_cabang              |nama_menu|total_penjualan|peringkat|
+-------------------------+---------+---------------+---------+
|Kopi Nusantara Magelang  |Menu-1   |8505000        |1        |
|Kopi Nusantara Magelang  |Menu-12  |8085000        |2        |
|Kopi Nusantara Semarang  |Menu-13  |8750000        |1        |
|Kopi Nusantara Semarang  |Menu-9   |8715000        |2        |
|Kopi Nusantara Yogyakarta|Menu-12  |9275000        |1        |
|Kopi Nusantara Yogyakarta|Menu-15  |8160000        |2        |
+-------------------------+---------+---------------+---------+



D. TRANSFORM — Spark SQL

In [11]:
df_gabungan.createOrReplaceTempView("gabungan")

penjualan_barista = spark.sql("""
     SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
     FROM gabungan
     GROUP BY kepala_barista
     ORDER BY total_penjualan DESC
""")

penjualan_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



E. LOAD

In [15]:
!hdfs dfs -mkdir -p/user/mahasiswa/tugas7/processed

df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
)

print("Laporan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori_menu.")
!hdfs dfs -ls/user/mahasiswa/tugas7/processed/laporan

#Verifikası: baca ulang lalu bandingkan jumlah baris
df_verifikasi = spark.read. parquet("hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan")
print("Jumlah baris data gabungan (sebelum disimpan):", df_gabungan.count())
print("Jumlah baris Parquet (setelah dibaca ulang):", df_verifikasi.count())

-mkdir: Illegal option -p/user/mahasiswa/tugas7/processed
Usage: hadoop fs [generic options]
	[-appendToFile [-n] <localsrc> ... <dst>]
	[-cat [-ignoreCrc] <src> ...]
	[-checksum [-v] <src> ...]
	[-chgrp [-R] GROUP PATH...]
	[-chmod [-R] <MODE[,MODE]... | OCTALMODE> PATH...]
	[-chown [-R] [OWNER][:[GROUP]] PATH...]
	[-concat <target path> <src path> <src path> ...]
	[-copyFromLocal [-f] [-p] [-l] [-d] [-t <thread count>] [-q <thread pool queue size>] <localsrc> ... <dst>]
	[-copyToLocal [-f] [-p] [-crc] [-ignoreCrc] [-t <thread count>] [-q <thread pool queue size>] <src> ... <localdst>]
	[-count [-q] [-h] [-v] [-t [<storage type>]] [-u] [-x] [-e] [-s] <path> ...]
	[-cp [-f] [-p | -p[topax]] [-d] [-t <thread count>] [-q <thread pool queue size>] <src> ... <dst>]
	[-createSnapshot <snapshotDir> [<snapshotName>]]
	[-deleteSnapshot <snapshotDir> <snapshotName>]
	[-df [-h] [<path> ...]]
	[-du [-s] [-h] [-v] [-x] <path> ...]
	[-expunge [-immediate] [-fs <path>]]
	[-find <path> ... <expressio

[Stage 34:>                                                         (0 + 1) / 1]

Laporan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori_menu.


-ls/user/mahasiswa/tugas7/processed/laporan: Unknown command
Usage: hadoop fs [generic options]
	[-appendToFile [-n] <localsrc> ... <dst>]
	[-cat [-ignoreCrc] <src> ...]
	[-checksum [-v] <src> ...]
	[-chgrp [-R] GROUP PATH...]
	[-chmod [-R] <MODE[,MODE]... | OCTALMODE> PATH...]
	[-chown [-R] [OWNER][:[GROUP]] PATH...]
	[-concat <target path> <src path> <src path> ...]
	[-copyFromLocal [-f] [-p] [-l] [-d] [-t <thread count>] [-q <thread pool queue size>] <localsrc> ... <dst>]
	[-copyToLocal [-f] [-p] [-crc] [-ignoreCrc] [-t <thread count>] [-q <thread pool queue size>] <src> ... <localdst>]
	[-count [-q] [-h] [-v] [-t [<storage type>]] [-u] [-x] [-e] [-s] <path> ...]
	[-cp [-f] [-p | -p[topax]] [-d] [-t <thread count>] [-q <thread pool queue size>] <src> ... <dst>]
	[-createSnapshot <snapshotDir> [<snapshotName>]]
	[-deleteSnapshot <snapshotDir> <snapshotName>]
	[-df [-h] [<path> ...]]
	[-du [-s] [-h] [-v] [-x] <path> ...]
	[-expunge [-immediate] [-fs <path>]]
	[-find <path> ... <expres

F. Ringkasan Eksekutif & Rekomendasi 

In [17]:
ringkasan_eksekutif = df_verifikasi.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")

).orderBy(col("total_penjualan").desc())

ringkasan_eksekutif.show(truncate=False)

+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|nama_cabang              |kota      |kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Semarang  |Semarang  |Nadia         |90542000       |1393            |64997.8463747308         |
|Kopi Nusantara Magelang  |Magelang  |Nadia         |87138000       |1302            |66926.26728110599        |
|Kopi Nusantara Yogyakarta|Yogyakarta|Reza          |84645000       |1305            |64862.06896551724        |
+-------------------------+----------+--------------+---------------+----------------+-------------------------+



KESIMPULAN : Berdasarkan tabel yang diperoleh, Kopi Nusantara Semarang menjadi cabang dengan kinerja terbaik karena mencatat penjualan tertinggi sebesar Rp90.542.000 dengan total 1.393 transaksi. Sementara itu, Kopi Nusantara Yogyakarta memiliki penjualan terendah, yaitu sebesar Rp84.645.000. Dari hasil Window Function, diketahui bahwa Menu-12 menjadi menu dengan penjualan tertinggi di Yogyakarta sebesar Rp9.275.000, disusul Menu-15 sebesar Rp8.160.000. Oleh karena itu, Menu-12 bisa lebih dipromosikan di cabang Yogyakarta karena memiliki penjualan paling tinggi dibandingkan menu lainnya di cabang tersebut. Promosi dapat dilakukan dengan menawarkan paket menu, memberikan diskon untuk pembelian tertentu, atau menjadikan Menu-12 sebagai rekomendasi utama kepada pelanggan. Dengan cara ini, diharapkan penjualan cabang Yogyakarta dapat meningkat dan mampu bersaing dengan cabang lainnya. Selain itu, hasil penjualan perlu dipantau secara berkala untuk mengetahui apakah strategi promosi yang dilakukan sudah efektif.

G. EXPLOR

In [18]:
df_gabungan.select("qty","harga", "total_penjualan").describe().show()

+-------+------------------+-----------------+-----------------+
|summary|               qty|            harga|  total_penjualan|
+-------+------------------+-----------------+-----------------+
|  count|              4000|             4000|             4000|
|   mean|           2.48475|          26348.5|         65581.25|
| stddev|1.1276459257379772|6547.882592228606|34912.95070556545|
|    min|                 1|            18000|            18000|
|    max|                 4|            35000|           140000|
+-------+------------------+-----------------+-----------------+



In [19]:
df_gabungan.groupBy("nama_cabang")\
           .pivot("kategori_menu")\
           .agg(spark_sum("total_penjualan"))\
           .show(truncate=False)

+-------------------------+--------+--------------+--------+
|nama_cabang              |Kopi    |Makanan Ringan|Non-Kopi|
+-------------------------+--------+--------------+--------+
|Kopi Nusantara Yogyakarta|21016000|45049000      |18580000|
|Kopi Nusantara Magelang  |21152000|48190000      |17796000|
|Kopi Nusantara Semarang  |22078000|49518000      |18946000|
+-------------------------+--------+--------------+--------+



In [20]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
